# 📢 Spam Detection Model

Train a classifier to identify spam/promotional reviews.

**Algorithms**: Random Forest, SVM
**Input**: Review text
**Output**: Spam/Not Spam classification


In [ ]:
!pip install -q scikit-learn pandas numpy nltk matplotlib seaborn joblib

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import nltk
import joblib
import warnings
warnings.filterwarnings('ignore')

nltk.download('punkt')
nltk.download('stopwords')

print('✅ All libraries imported')

## 📊 Load Training Data

Dataset: `review` and `is_spam` (0=Not Spam, 1=Spam)

In [ ]:
sample_data = {
    'review': [
        'Great product, very happy',           # Not spam
        'Works as described',                   # Not spam
        'BUY NOW LIMITED OFFER!!!',            # Spam
        'CLICK HERE TO PURCHASE',              # Spam
        'Excellent customer service',          # Not spam
        'Check out my website link',           # Spam
        'Very satisfied with quality',         # Not spam
        'You wont believe this offer',         # Spam
        'Fast shipping, great packaging',      # Not spam
        'Visit our store for more',            # Spam
    ],
    'is_spam': [0, 0, 1, 1, 0, 1, 0, 1, 0, 1]  # 0=Not Spam, 1=Spam
}

df = pd.DataFrame(sample_data)
print('Dataset shape:', df.shape)
print('\nSpam distribution:')
print(df['is_spam'].value_counts())
print('\nSample data:')
print(df.head())

## 🧹 Preprocessing

In [ ]:
import re

def preprocess_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['review_clean'] = df['review'].apply(preprocess_text)
print('✅ Preprocessing complete')

## 🔢 Vectorization

In [ ]:
vectorizer = TfidfVectorizer(max_features=100, ngram_range=(1, 2))
X = vectorizer.fit_transform(df['review_clean'])
y = df['is_spam']

print(f'Feature matrix shape: {X.shape}')

## 📚 Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Training samples: {X_train.shape[0]}')
print(f'Test samples: {X_test.shape[0]}')

## 🤖 Train Models

In [ ]:
# Random Forest
print('Training Random Forest...')
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)
print(f'✅ Random Forest Accuracy: {rf_acc:.4f}')

# SVM
print('\nTraining SVM...')
svm_model = SVC(kernel='rbf', probability=True, random_state=42)
svm_model.fit(X_train, y_train)
svm_pred = svm_model.predict(X_test)
svm_acc = accuracy_score(y_test, svm_pred)
print(f'✅ SVM Accuracy: {svm_acc:.4f}')

## 📊 Evaluation

In [ ]:
print('='*60)
print('RANDOM FOREST - PRIMARY MODEL')
print('='*60)
print(classification_report(y_test, rf_pred, target_names=['Not Spam', 'Spam']))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, rf_pred))

comparison = pd.DataFrame({
    'Model': ['Random Forest', 'SVM'],
    'Accuracy': [rf_acc, svm_acc]
})
print('\nModel Comparison:')
print(comparison.sort_values('Accuracy', ascending=False).to_string(index=False))

## 💾 Save Model

In [ ]:
joblib.dump(rf_model, 'spam_model.pkl')
print('✅ Spam model saved!')
print('   - spam_model.pkl')

## 🧪 Test

In [ ]:
test_reviews = [
    'Great quality and fast delivery',
    'CLICK HERE FOR BEST DEALS!!!',
    'Excellent product'
]

print('Testing:')
for review in test_reviews:
    review_clean = preprocess_text(review)
    X_new = vectorizer.transform([review_clean])
    pred = rf_model.predict(X_new)[0]
    label = 'SPAM' if pred == 1 else 'NOT SPAM'
    print(f'{review} → {label}')